# GO Reasoning Dataset

### Step 1: Make a fasta file of all proteins in our train + test set

In [1]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset

In [2]:
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

all = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all = all.drop(columns=['protein_names','protein_function','organism','length','subcellular_location','go_ids','go_bp','go_mf','go_cc','string_id','interaction_partners','full_interaction_info','interpro_ids','interpro_location','structure_path'])
all.drop_duplicates(inplace=True)
all.reset_index(drop=True, inplace=True)

README.md: 0.00B [00:00, ?B/s]

cafa5_reasoning/train-00000-of-00002.par(…):   0%|          | 0.00/104M [00:00<?, ?B/s]

cafa5_reasoning/train-00001-of-00002.par(…):   0%|          | 0.00/77.7M [00:00<?, ?B/s]

cafa5_reasoning/test-00000-of-00001.parq(…):   0%|          | 0.00/393k [00:00<?, ?B/s]

cafa5_reasoning/test_superset-00000-of-0(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133534 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/264 [00:00<?, ? examples/s]

Generating test_superset split:   0%|          | 0/141847 [00:00<?, ? examples/s]

In [3]:
all

,protein_id,sequence
0,A0A087X1C5,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...
1,A0A0B4J2F0,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...
2,A0A0A7EPL0,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...
3,A0A0B4J1G0,MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...
4,A0A0B4J1N3,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...
...,...,...
201752,Q4A3R3,MGTSAVILEICLLLSQVLTTVSSTTQTESTTEDRTQITETAFWETQ...
201753,Q5E9L2,MNWRFVELLYFLFVWGRISVQPSHQEPAATDQHVSKEFDWLISDRG...
201754,Q3T0K9,MNSKGQYPTQPTYPVQPPGNPVYPQTLHLPQAPPYTDAPPAYSELY...
201755,Q58CX7,MEPRLGPKAAALHLGWPFLLLWVSGLSYSVSSPASPSPSPVSRVRT...


In [4]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Convert DataFrame rows to SeqRecord objects
records = [
    SeqRecord(Seq(row['sequence']), id=row['protein_id'], description="")
    for _, row in all.iterrows()
]

# Write to FASTA file
with open("all_proteins.fasta", "w") as output_handle:
    SeqIO.write(records, output_handle, "fasta")

### Step 2: BLAST these proteins against nr and get the top 10 hits

In [ ]:
time diamond blastp \
  -q all_proteins.fasta \
  -d <path_to_database>/nr.dmnd  \
  --masking 0 \
  --sensitive -k 25 \
  -b 10 -c1 \
  --unal 1 \
  --threads 64 \
  -f 6 qseqid  qstart qend qlen qstrand \
       sseqid  sstart send slen \
       pident evalue \
       staxids sscinames \
       stitle \
  -o all_cafa_proteins.tsv

In [1]:
import pandas as pd
import numpy as np

In [4]:
all_cafa_proteins = pd.read_csv('all_cafa_proteins.tsv', sep='\t', 
                                names=['cafa5_protein', 'q_start', 'q_end','q_length', 'strand',
                                      'blast_protein','t_start','t_end', 't_length','pident','evalue','taxid','species','protein_name'])

In [5]:
all_cafa_proteins

,cafa5_protein,q_start,q_end,q_length,strand,blast_protein,t_start,t_end,t_length,pident,evalue,taxid,species,protein_name
0,A0A087X1C5,1,515,515,+,A0A087X1C5.1,1,515,515,100.0,0.0,9606.0,Homo sapiens,A0A087X1C5.1 PUTATIVE PSEUDOGENE: RecName: Ful...
1,A0A087X1C5,1,515,515,+,NP_001335315.1,1,516,516,99.8,0.0,9606.0,Homo sapiens,NP_001335315.1 putative cytochrome P450 2D7 [H...
2,A0A087X1C5,1,515,515,+,AAO49806.1,1,516,516,98.6,0.0,9606.0,Homo sapiens,AAO49806.1 cytochrome P450 [Homo sapiens]
3,A0A087X1C5,1,515,515,+,AEU08335.1,1,497,497,92.0,0.0,9606.0,Homo sapiens,AEU08335.1 CYP2D6 [Homo sapiens]
4,A0A087X1C5,1,515,515,+,XP_034805195.3,1,497,497,92.2,0.0,9597.0,Pan paniscus,XP_034805195.3 cytochrome P450 2D6 isoform X1 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4940545,A7E300,34,1112,1112,+,XP_027386041.1,445,1523,1523,99.5,0.0,30522.0,Bos indicus x Bos taurus,XP_027386041.1 rho GTPase-activating protein 7...
4940546,A7E300,33,1112,1112,+,XP_027386044.1,84,1163,1163,99.4,0.0,30522.0,Bos indicus x Bos taurus,XP_027386044.1 rho GTPase-activating protein 7...
4940547,A7E300,37,1112,1112,+,ELR59854.1,1,1076,1076,99.8,0.0,72004.0,Bos mutus,"ELR59854.1 Rho GTPase-activating protein 7, pa..."
4940548,A7E300,34,1112,1112,+,XP_010837132.1,445,1523,1523,99.4,0.0,43346.0,Bison bison bison,XP_010837132.1 PREDICTED: rho GTPase-activatin...


In [6]:
all_cafa_proteins['protein_name'] = all_cafa_proteins['protein_name'].str.split('>').str[0].str.strip()

In [7]:
from datasets import Dataset

# Create Hugging Face Datasets
blast_results = Dataset.from_pandas(all_cafa_proteins, preserve_index=False)

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from datasets import DatasetDict

blast_dataset = DatasetDict({
    "metadata": blast_results
})

In [9]:
blast_dataset

DatasetDict({
    metadata: Dataset({
        features: ['cafa5_protein', 'q_start', 'q_end', 'q_length', 'strand', 'blast_protein', 't_start', 't_end', 't_length', 'pident', 'evalue', 'taxid', 'species', 'protein_name'],
        num_rows: 4940550
    })
})

In [ ]:
blast_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="blast_metadata",
    commit_message="Uploading the BLAST results. 25 proteins per cafa5_protein"
)

### Step 3: Format the PPI Data

Currently the PPI data does not include the Protein names, so I need to include that information when we are creating the data for the model

Download the dataset with all of the info about the proteins

In [ ]:
wget https://stringdb-downloads.org/download/protein.info.v12.0.txt.gz

In [ ]:
gunzip protein.info.v12.0.txt.gz

In [ ]:
wget https://string-db.org/mapping_files/uniprot/all_organisms.uniprot_2_string.tsv

Get the PPI Partners

In [1]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset

In [37]:
ds = load_dataset("wanglab/cafa5", "swissprot_reasoning")
swissprot_train = ds["train"].to_pandas()
swissprot_val = ds["val"].to_pandas()

In [38]:
swissprot_train['species_string'] = swissprot_train['string_id'].apply(
    lambda x: x.split('.')[0] if isinstance(x, str) and '.' in x else None
)
swissprot_val['species_string'] = swissprot_val['string_id'].apply(
    lambda x: x.split('.')[0] if isinstance(x, str) and '.' in x else None
)

Load in STRING Data

In [4]:
string_info = pd.read_csv('<path_to_string_data>/protein.info.v12.0.txt', sep='\t')

In [5]:
string_info['species'] = string_info['#string_protein_id'].str.split('.').str[0]
string_info['string_name'] = string_info['annotation'].apply(lambda x: x.split('; ', 1)[0])
string_info['string_description'] = string_info['annotation'].apply(lambda x: x.split('; ', 1)[1] if '; ' in x else None)

In [6]:
string_info[100:110]

,#string_protein_id,preferred_name,protein_size,annotation,species,string_name,string_description
100,23.BEL05_00560,OEG74124.1,168,Hypothetical protein; Derived by automated com...,23,Hypothetical protein,Derived by automated computational analysis us...
101,23.BEL05_00565,OEG74125.1,1003,Collagenase; Derived by automated computationa...,23,Collagenase,Derived by automated computational analysis us...
102,23.BEL05_00570,OEG74126.1,338,NAD(P)H-quinone oxidoreductase; Derived by aut...,23,NAD(P)H-quinone oxidoreductase,Derived by automated computational analysis us...
103,23.BEL05_00575,OEG74127.1,142,Hypothetical protein; Derived by automated com...,23,Hypothetical protein,Derived by automated computational analysis us...
104,23.BEL05_00580,OEG74128.1,345,Galactose mutarotase; Converts alpha-aldose to...,23,Galactose mutarotase,Converts alpha-aldose to the beta-anomer.
105,23.BEL05_00585,galK,388,Galactokinase; Catalyzes the transfer of the g...,23,Galactokinase,Catalyzes the transfer of the gamma-phosphate ...
106,23.BEL05_00590,OEG74130.1,347,Galactose-1-phosphate uridylyltransferase; Der...,23,Galactose-1-phosphate uridylyltransferase,Derived by automated computational analysis us...
107,23.BEL05_00595,OEG74131.1,475,Sodium:solute symporter; Derived by automated ...,23,Sodium:solute symporter,Derived by automated computational analysis us...
108,23.BEL05_00600,OEG74132.1,1055,Beta-galactosidase; Derived by automated compu...,23,Beta-galactosidase,Derived by automated computational analysis us...
109,23.BEL05_00605,OEG74282.1,310,Hypothetical protein; Derived by automated com...,23,Hypothetical protein,Derived by automated computational analysis us...


Uniprot to STRING mapping

In [7]:
import pandas as pd

mapping_path = "<path_to_string_data>/all_organisms.uniprot_2_string.tsv"
cols = ["species", "uniprot_ac|uniprot_id", "string_id", "identity", "bit_score"]

# Load the file, skipping comment lines (those starting with '#') and assigning custom column names
uniprot_string_map_df = pd.read_csv(
    mapping_path,
    sep="\t",
    comment="#",
    header=None,
    names=cols,
    on_bad_lines='skip',
    engine='python'
)

In [8]:
# Split the combined UniProt column into two: accession and ID
uniprot_string_map_df[['uniprot_ac', 'uniprot_id']] = uniprot_string_map_df['uniprot_ac|uniprot_id'].str.split('|', n=1, expand=True)

# Drop the original combined column
uniprot_string_map_df.drop(columns=['uniprot_ac|uniprot_id'], inplace=True)

Create the extra metadata row in 'all'

In [9]:
uniprot_string_map_df

,species,string_id,identity,bit_score,uniprot_ac,uniprot_id
0,5874,5874.P16019,Src,Src,P16019,HSP70_THEAN
1,5874,5874.P25781,Src,Src,P25781,CYSP_THEAN
2,5874,5874.Q26671,Src,Src,Q26671,CDC2H_THEAN
3,5874,5874.Q27399,Src,Src,Q27399,Q27399_THEAN
4,5874,5874.Q37679,Src,Src,Q37679,COX3_THEAN
...,...,...,...,...,...,...
46131357,1030841,1030841.HMPREF9370_0777,100.0,198.0,G4CNW8,G4CNW8_9NEIS
46131358,1030841,1030841.HMPREF9370_0925,100.0,419.5,G4CPB5,G4CPB5_9NEIS
46131359,1030841,1030841.HMPREF9370_1435,100.0,771.9,G4CQS5,G4CQS5_9NEIS
46131360,1030841,1030841.HMPREF9370_1692,100.0,592.8,G4CRI2,G4CRI2_9NEIS


In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Fast build: (species, preferred_name) -> string_id
name_to_string_id_lookup = {
    (str(species), preferred_name): string_id
    for species, preferred_name, string_id in zip(
        string_info['species'],
        string_info['preferred_name'],
        string_info['#string_protein_id']
    )
}

# Fast build: (species, string_id) -> UniProt metadata
string_id_to_uniprot_metadata = {
    (str(species), string_id): {
        'string_protein_id': string_id,
        'uniprot_ac': uniprot_ac,
        'uniprot_id': uniprot_id,
        'identity': identity,
        'bit_score': bit_score
    }
    for species, string_id, uniprot_ac, uniprot_id, identity, bit_score in zip(
        uniprot_string_map_df['species'],
        uniprot_string_map_df['string_id'],
        uniprot_string_map_df['uniprot_ac'],
        uniprot_string_map_df['uniprot_id'],
        uniprot_string_map_df['identity'],
        uniprot_string_map_df['bit_score']
    )
}

In [11]:
type(swissprot_train['full_interaction_info'][2])

numpy.ndarray

In [45]:
def collect_metadata_flat(df):
    metadata_rows = []

    valid_mask = df['interaction_partners'].apply(lambda x: isinstance(x, np.ndarray))
    valid_indices = df.index[valid_mask]

    for idx in tqdm(valid_indices, desc="Collecting UniProt metadata"):
        partners = df.at[idx, 'interaction_partners']
        interaction_info = df.at[idx, 'full_interaction_info']
        species = str(df.at[idx, 'species_string'])
        swissprot_protein = df.at[idx, 'protein_id']

        # Safety check: make sure the interaction_info is a list and matches the partner list
        if not isinstance(interaction_info, np.ndarray) or len(interaction_info) != len(partners):
            print("something gone wrong at ",idx)

        for partner, info in zip(partners, interaction_info):
            string_id = name_to_string_id_lookup.get((species, partner))
            if string_id:
                uniprot_metadata = string_id_to_uniprot_metadata.get((species, string_id))
                if uniprot_metadata:
                    metadata_rows.append({
                        'swissprot_protein': swissprot_protein,
                        'partner_id': partner,
                        **uniprot_metadata,
                        **info  # includes combined_score, coexpression, experimental, etc.
                    })

    return pd.DataFrame(metadata_rows)

In [46]:
train_metadata_df = collect_metadata_flat(swissprot_train)
val_metadata_df = collect_metadata_flat(swissprot_val)

In [47]:
# Optionally combine all:
all_metadata_df = pd.concat([train_metadata_df, val_metadata_df], ignore_index=True)
all_metadata_df = all_metadata_df.drop_duplicates()

In [48]:
all_metadata_df

,swissprot_protein,partner_id,string_protein_id,uniprot_ac,uniprot_id,identity,bit_score,coexpression,combined_score,cooccurence,database,experimental,fusion,neighborhood,protein2,textmining
0,A9I8F5,murB,94624.Bpet1546,A9IGA7,A9IGA7_BORPD,100.0,707.2,0,916,165,0,0,0,95,murB,899
1,A9I8F5,fis,94624.Bpet0907,A9I8G0,A9I8G0_BORPD,100.0,161.8,86,809,0,0,0,0,780,fis,130
2,A9I8F5,dusB,94624.Bpet0908,A9I8G2,A9I8G2_BORPD,100.0,700.7,0,766,179,0,0,0,728,dusB,0
3,A9I8F5,polA,94624.Bpet3126,A9ITV6,A9ITV6_BORPD,100.0,1867.0,0,722,251,0,0,0,367,polA,462
4,A9I8F5,ruvA,94624.Bpet0904,A9I8F2,RUVA_BORPD,100.0,365.2,767,994,561,0,715,0,737,ruvA,401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6569391,B0C1Z3,glpX-2,329726.AM1_5758,B0BZF8,FBSB2_ACAM1,100.0,682.2,0,900,48,900,0,0,0,glpX-2,0
6569392,B0C1Z3,tal-3,329726.AM1_6318,B0C7G4,B0C7G4_ACAM1,100.0,1885.2,0,938,0,900,0,0,126,tal-3,353
6569393,B0C1Z3,fbp,329726.AM1_2875,B0CAD9,F16A1_ACAM1,100.0,716.5,0,913,0,900,0,0,0,fbp,171
6569394,B0C1Z3,fbp-2,329726.AM1_6321,B0C7G7,F16A2_ACAM1,100.0,661.0,0,909,0,900,0,0,0,fbp-2,136


### Testing what are the PPI partners that I am losing

In [49]:
all = pd.concat([swissprot_train, swissprot_val])
all.reset_index(drop=True, inplace=True)

In [50]:
import numpy as np

# Only keep rows where interaction_partners is a list
valid_mask = all['interaction_partners'].apply(lambda x: isinstance(x, np.ndarray))
filtered_df = all.loc[valid_mask, ['protein_id', 'interaction_partners']]

# Flatten using list comprehension
partner_pairs = [
    (protein_id, partner)
    for protein_id, partners in zip(filtered_df['protein_id'], filtered_df['interaction_partners'])
    for partner in partners
]

# Convert to DataFrame
partner_pairs_df = pd.DataFrame(partner_pairs, columns=['swissprot_protein', 'partner_id'])
partner_pairs_df = partner_pairs_df.drop_duplicates()

In [51]:
# Step 2: Check which are missing from all_metadata_df
# Use an inner merge to find matches
merged = partner_pairs_df.merge(
    all_metadata_df[['swissprot_protein', 'partner_id']],
    on=['swissprot_protein', 'partner_id'],
    how='left',
    indicator=True
)

# Filter missing entries
missing = merged[merged['_merge'] == 'left_only']

# Summary
print(f"Total partner pairs in 'all': {len(partner_pairs_df)}")
print(f"Found in metadata: {len(partner_pairs_df) - len(missing)}")
print(f"Missing from metadata: {len(missing)}")

# Optional: view missing entries
# print(missing.head())

Total partner pairs in 'all': 6612764
Found in metadata: 6569396
Missing from metadata: 43368


In [52]:
missing

,swissprot_protein,partner_id,_merge
2242,P30116,ENSMAUP00000017955,left_only
2269,P30116,ENSMAUP00000013857,left_only
2270,P30116,Gsr,left_only
2428,A1VJ34,Pnap_3647,left_only
2695,Q2YM12,tuf-2-2,left_only
...,...,...,...
6611596,Q15TZ4,Patl_3996,left_only
6611832,A1KB21,tufB,left_only
6612148,Q5ZYN8,tufB,left_only
6612228,P49623,CDG14928.1,left_only


In [53]:
# Step 1: Count total partners per swissprot_protein in `all`
partner_counts_all = (
    all[all['interaction_partners'].apply(lambda x: isinstance(x, np.ndarray))]
    .assign(total_partners=lambda df: df['interaction_partners'].apply(len))
    [['protein_id', 'total_partners']]
    .rename(columns={'protein_id': 'swissprot_protein'})
)
partner_counts_all.drop_duplicates()
len(partner_counts_all)

# Step 2: Count number of missing partners per protein
missing_counts = (
    missing.groupby('swissprot_protein')
    .size()
    .reset_index(name='missing_partners')
)

# Step 3: Merge both counts
summary_df = partner_counts_all.merge(missing_counts, on='swissprot_protein', how='inner')

# Optional: Sort by number of missing partners
summary_df = summary_df.sort_values(by='total_partners', ascending=False)

summary_df.head()

,swissprot_protein,total_partners,missing_partners
5435,P27785,759,496
15433,C0H859,749,181
7097,P47839,694,220
11064,P68200,679,37
15099,P62978,675,7


In [54]:
summary_df

,swissprot_protein,total_partners,missing_partners
5435,P27785,759,496
15433,C0H859,749,181
7097,P47839,694,220
11064,P68200,679,37
15099,P62978,675,7
...,...,...,...
6303,P46365,1,1
12669,R9UAM1,1,1
15282,P65814,1,1
6298,P0A0N5,1,1


In [55]:
summary_df['diff'] = summary_df['total_partners'] - summary_df['missing_partners']

In [56]:
len(summary_df[summary_df['diff'] == 0])

77

In [57]:
summary_df.sort_values(by='diff', ascending=False)

,swissprot_protein,total_partners,missing_partners,diff
15099,P62978,675,7,668
11394,P0CG54,672,7,665
7704,P63052,660,4,656
11064,P68200,679,37,642
8995,A5JST6,620,3,617
...,...,...,...,...
10762,P32197,3,3,0
2183,P10572,5,5,0
9034,P56948,5,5,0
13438,P83456,5,5,0


77 protein out of 400k will be losing all of their PPI partners. I am fine with that

## Fetching Uniprot info

In [87]:
import requests
import pandas as pd
from io import StringIO
from time import sleep
from tqdm import tqdm

def fetch_uniprot_metadata(all_accessions, output_file, deleted_file, batch_size=100):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    fields = "accession,protein_name,cc_function,cc_subcellular_location"
    first = True
    deleted = []

    # Split into batches
    batches = [all_accessions[i:i + batch_size] for i in range(0, len(all_accessions), batch_size)]

    for batch in tqdm(batches, desc="Processing batches"):
        # Correct query: use 'accession:ACC OR accession:ACC ...'
        query = " OR ".join([f"accession:{acc}" for acc in batch])
        params = {"query": query, "format": "tsv", "fields": fields}

        for attempt in range(3):  # Retry up to 3 times
            try:
                r = requests.get(base_url, params=params, timeout=60)
                if r.status_code == 429:  # Rate limit
                    print("Rate limited, sleeping 10s...")
                    sleep(10)
                    continue
                r.raise_for_status()

                df = pd.read_csv(StringIO(r.text), sep="\t")

                # Compare returned vs batch to identify deleted/missing
                returned = set(df["Entry"].tolist())
                missing = set(batch) - returned
                deleted.extend(missing)

                # Write output
                df.to_csv(output_file, sep="\t", index=False, header=first, mode="a")
                first = False
                break  # Success, break retry loop

            except Exception as e:
                print(f"Error on attempt {attempt+1}: {e}")
                sleep(5)
                if attempt == 2:
                    # If all retries fail, mark entire batch as missing
                    deleted.extend(batch)

        sleep(0.1)  # Rate-limit buffer

    # Save deleted accessions
    with open(deleted_file, "w") as f:
        for acc in deleted:
            f.write(acc + "\n")

    print(f"✅ Done. Saved to {output_file}, deleted proteins saved to {deleted_file}")

In [88]:
fetch_uniprot_metadata(all_metadata_df['uniprot_ac'].unique(), 
                             output_file="ppi_uniprot_metadata_swissprot.tsv", deleted_file="ppi_deleted_accessions_swissprot.txt")

Processing batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8351/8351 [1:50:40<00:00,  1.26it/s]


✅ Done. Saved to ppi_uniprot_metadata_swissprot.tsv, deleted proteins saved to ppi_deleted_accessions_swissprot.txt


In [89]:
uniprot = pd.read_csv('ppi_uniprot_metadata_swissprot.tsv', sep='\t')
uniprot = uniprot.rename(columns={'Entry' : 'uniprot_ac'})

In [90]:
uniprot

,uniprot_ac,Protein names,Function [CC],Subcellular location [CC]
0,Q2JQT5,DNA-directed RNA polymerase subunit gamma (RNA...,FUNCTION: DNA-dependent RNA polymerase catalyz...,NaN
1,Q2JQX5,Adenylosuccinate synthetase (AMPSase) (AdSS) (...,FUNCTION: Plays an important role in the de no...,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000255|H...
2,Q2JRD6,Glutamate--tRNA ligase (EC 6.1.1.17) (Glutamyl...,FUNCTION: Catalyzes the attachment of glutamat...,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000255|H...
3,Q2JSV7,ATP synthase subunit c (ATP synthase F(0) sect...,FUNCTION: F(1)F(0) ATP synthase produces ATP f...,SUBCELLULAR LOCATION: Cellular thylakoid membr...
4,Q2JUX4,Elongation factor Tu (EF-Tu) (EC 3.6.5.3),FUNCTION: GTP hydrolase that promotes the GTP-...,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000255|H...
...,...,...,...,...
210047,Q3J6E3,Anti-sigma regulatory kinase (EC 2.7.11.1),NaN,NaN
210048,Q8XMA0,Pirin-like protein,NaN,NaN
210049,A0A0H2V9Y8,Hypothetical transcriptional regulator ydiP,NaN,NaN
210050,A5UKA0,Glycosyltransferase,NaN,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...


## Merging it with the PPI data

In [91]:
all_metadata_df = all_metadata_df.merge(uniprot, on='uniprot_ac', how='left')

In [95]:
string_info[string_info['#string_protein_id']=='7227.FBpp0303878']

,#string_protein_id,preferred_name,protein_size,annotation,species,string_name,string_description
4362277,7227.FBpp0303878,nSyb,206,"Neuronal synaptobrevin, isoform J; Neuronal Sy...",7227,"Neuronal synaptobrevin, isoform J",Neuronal Synaptobrevin (nSyb) encodes a SNAP r...


In [98]:
from tqdm import tqdm
import pandas as pd

# Step 1: Build lookup dictionaries from string_info
name_lookup = dict(zip(string_info['#string_protein_id'], string_info['string_name']))
desc_lookup = dict(zip(string_info['#string_protein_id'], string_info['string_description']))

In [99]:
# Step 2: Mask rows with missing or 'merged' protein names
mask = (all_metadata_df['Protein names'].isna()) | (all_metadata_df['Protein names'] == 'merged')
indices = all_metadata_df[mask].index

# Step 3: Update those rows in-place using tqdm for a progress bar
for idx in tqdm(indices, desc="Updating protein names and descriptions"):
    protein_id = all_metadata_df.at[idx, 'string_protein_id']
    
    if protein_id in name_lookup:
        all_metadata_df.at[idx, 'Protein names'] = name_lookup[protein_id]
    if protein_id in desc_lookup:
        all_metadata_df.at[idx, 'Function [CC]'] = desc_lookup[protein_id]

Updating protein names and descriptions: 100%|████████████████████████████████████████████████████████████████████████████████████████| 4642561/4642561 [04:06<00:00, 18867.24it/s]


In [115]:
import re
import pandas as pd

# ------------------------------
# Precompile regex patterns
# ------------------------------
includes_pattern = re.compile(r'\s*\[Includes:[^\]]*\]', re.IGNORECASE)
cleaved_pattern = re.compile(r'\s*\[Cleaved into:[^\]]*\]', re.IGNORECASE)

location_pattern = re.compile(
    r',\s*(mitochondrial|chloroplastic|cytoplasmic|cytosolic|nuclear|peroxisomal|endoplasmic\s+reticulum|'
    r'plasma\s+membrane|cell\s+wall|vacuolar|ribosomal|membrane|extracellular|lysosomal|golgi)\.?$',
    re.IGNORECASE
)

ec_pattern = re.compile(r'\s*\(EC\s+[\d\.\-n]+\)', re.IGNORECASE)

cofactors = [
    'ubiquinone', 'NADP', 'NAD', 'NADH', 'NADPH', 'FAD', 'FMN', 'ATP', 'ADP', 'GTP', 'GDP',
    'Mn', 'Cu-Zn', 'Fe', 'Zn', 'Mg', 'Ca', 'Mo', 'Ni', 'Co',
    'ADP-forming', 'UDP-forming', 'ADP/GDP-forming', 'GTP-forming',
    'glutamine-hydrolyzing', 'isomerizing', 'ammonia'
]
cofactor_pattern = re.compile(r'\s*[\[\(](?:' + '|'.join(map(re.escape, cofactors)) + r')[\]\)]', re.IGNORECASE)

long_paren_pattern_standard = re.compile(r'\s*\([^)]{30,}\)')
long_paren_pattern_chemical = re.compile(r'\s*\([^)]{40,}\)')

trailing_pattern = re.compile(r'[,\s]+$')
chemical_start_pattern = re.compile(r'^\([SREZ3456789]\w*\)-')

# ------------------------------
# Cleaning function
# ------------------------------
def clean_protein_name_safe(name):
    if pd.isna(name):
        return name

    name_str = str(name).strip()
    name_str = includes_pattern.sub('', name_str)
    name_str = cleaved_pattern.sub('', name_str)
    name_str = location_pattern.sub('', name_str)
    name_str = ec_pattern.sub('', name_str)
    name_str = cofactor_pattern.sub('', name_str)

    # Remove only long parentheses
    name_str = long_paren_pattern_standard.sub('', name_str)
    name_str = long_paren_pattern_chemical.sub('', name_str)

    # Remove trailing commas and extra spaces
    name_str = re.sub(r'\s+', ' ', name_str)
    name_str = trailing_pattern.sub('', name_str)

    return name_str.strip()

# ------------------------------
# Step 1: Build mapping for unique protein names
# ------------------------------
unique_names = all_metadata_df['Protein names'].unique()
cleaned_mapping = {}
for name in tqdm(unique_names, desc="Cleaning unique protein names"):
    cleaned_mapping[name] = clean_protein_name_safe(name)

# ------------------------------
# Step 2: Apply mapping to DataFrame
# ------------------------------
all_metadata_df['Protein names_cleaned'] = all_metadata_df['Protein names'].map(cleaned_mapping)

# ------------------------------
# Step 3: Compute basic stats
# ------------------------------
all_metadata_df['length_before'] = all_metadata_df['Protein names'].map(lambda x: len(str(x)) if pd.notna(x) else 0)
all_metadata_df['length_after'] = all_metadata_df['Protein names_cleaned'].map(lambda x: len(str(x)) if pd.notna(x) else 0)

stats = {
    'mean_length_before': all_metadata_df['length_before'].mean(),
    'mean_length_after': all_metadata_df['length_after'].mean(),
    'max_length_before': all_metadata_df['length_before'].max(),
    'max_length_after': all_metadata_df['length_after'].max(),
    'min_length_before': all_metadata_df['length_before'].min(),
    'min_length_after': all_metadata_df['length_after'].min(),
    'num_empty_after': (all_metadata_df['length_after'] == 0).sum()
}

print("Protein name cleaning stats:")
for k, v in stats.items():
    print(f"{k}: {v}")

Cleaning unique protein names: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 192009/192009 [00:03<00:00, 50553.13it/s]


Protein name cleaning stats:
mean_length_before: 46.503134165606944
mean_length_after: 40.059512899435454
max_length_before: 1259
max_length_after: 598
min_length_before: 2
min_length_after: 0
num_empty_after: 4


In [120]:
# Filling in the 4 that were missing
all_metadata_df.at[34789, 'Protein names_cleaned'] = 'Dehydrogenase with different specificities'
all_metadata_df.at[1573380, 'Protein names_cleaned'] = 'RND efflux membrane fusion protein'
all_metadata_df.at[3670682, 'Protein names_cleaned'] = 'ADP-heptose LPS heptosyltransferase II'
all_metadata_df.at[5715452, 'Protein names_cleaned'] = 'Dehydrogenase with different specificities'

In [121]:
all_metadata_df

,swissprot_protein,partner_id,string_protein_id,uniprot_ac,uniprot_id,identity,bit_score,coexpression,combined_score,cooccurence,...,fusion,neighborhood,protein2,textmining,Protein names,Function [CC],Subcellular location [CC],Protein names_cleaned,length_before,length_after
0,A9I8F5,murB,94624.Bpet1546,A9IGA7,A9IGA7_BORPD,100.0,707.2,0,916,165,...,0,95,murB,899,UDP-N-acetylenolpyruvoylglucosamine reductase ...,FUNCTION: Cell wall formation. {ECO:0000256|AR...,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000256|A...,UDP-N-acetylenolpyruvoylglucosamine reductase,96,45
1,A9I8F5,fis,94624.Bpet0907,A9I8G0,A9I8G0_BORPD,100.0,161.8,86,809,0,...,0,780,fis,130,DNA-binding protein fis,Belongs to the transcriptional regulatory Fis ...,NaN,DNA-binding protein fis,23,23
2,A9I8F5,dusB,94624.Bpet0908,A9I8G2,A9I8G2_BORPD,100.0,700.7,0,766,179,...,0,728,dusB,0,tRNA-dihydrouridine synthase B (EC 1.3.1.-),"FUNCTION: Catalyzes the synthesis of 5,6-dihyd...",NaN,tRNA-dihydrouridine synthase B,43,30
3,A9I8F5,polA,94624.Bpet3126,A9ITV6,A9ITV6_BORPD,100.0,1867.0,0,722,251,...,0,367,polA,462,DNA polymerase I,"In addition to polymerase activity, this DNA p...",NaN,DNA polymerase I,16,16
4,A9I8F5,ruvA,94624.Bpet0904,A9I8F2,RUVA_BORPD,100.0,365.2,767,994,561,...,0,737,ruvA,401,Holliday junction branch migration complex sub...,FUNCTION: The RuvA-RuvB-RuvC complex processes...,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000255|H...,Holliday junction branch migration complex sub...,55,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6621215,B0C1Z3,glpX-2,329726.AM1_5758,B0BZF8,FBSB2_ACAM1,100.0,682.2,0,900,48,...,0,0,glpX-2,0,"D-fructose 1,6-bisphosphatase class 2/sedohept...",FUNCTION: Catalyzes the hydrolysis of fructose...,NaN,"D-fructose 1,6-bisphosphatase class 2/sedohept...",126,98
6621216,B0C1Z3,tal-3,329726.AM1_6318,B0C7G4,B0C7G4_ACAM1,100.0,1885.2,0,938,0,...,0,126,tal-3,353,Transaldolase,Transaldolase is important for the balance of ...,NaN,Transaldolase,13,13
6621217,B0C1Z3,fbp,329726.AM1_2875,B0CAD9,F16A1_ACAM1,100.0,716.5,0,913,0,...,0,0,fbp,171,"Fructose-1,6-bisphosphatase class 1 1 (FBPase ...",NaN,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000255|H...,"Fructose-1,6-bisphosphatase class 1 1 (FBPase ...",129,56
6621218,B0C1Z3,fbp-2,329726.AM1_6321,B0C7G7,F16A2_ACAM1,100.0,661.0,0,909,0,...,0,0,fbp-2,136,"Fructose-1,6-bisphosphatase class 1 2 (FBPase ...",NaN,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000255|H...,"Fructose-1,6-bisphosphatase class 1 2 (FBPase ...",129,56


## Edit the cafa5 data to reflect this update

In [123]:
swissprot_train = swissprot_train.drop(columns=['species_string','full_interaction_info'])
swissprot_val = swissprot_val.drop(columns=['species_string','full_interaction_info'])

In [124]:
# Step 1: Group partner_ids by cafa5_protein
grouped_partners = (
    all_metadata_df
    .groupby('swissprot_protein')['partner_id']
    .apply(list)
)

In [125]:
# Step 2: Assign
swissprot_train['interaction_partners'] = swissprot_train['protein_id'].map(grouped_partners)
swissprot_val['interaction_partners'] = swissprot_val['protein_id'].map(grouped_partners)

In [126]:
swissprot_train

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,interpro_ids,interpro_location,string_id,interaction_partners
0,B4TU36,UPF0391 membrane protein YtjA,None,Salmonella schwarzengrund (strain CVM19633),53,Cell membrane,MFRWGIIFLVIALIAAALGFGGLAGTAAGAAKIVFVVGIVLFLVSL...,"[GO:0005575, GO:0110165, GO:0016020, GO:0005886]",None,None,"[GO:0005575, GO:0110165, GO:0016020, GO:0005886]",AF-B4TU36-F1-model_v4.cif,[IPR009760],"{""IPR009760"": [1, 53]}",None,NaN
1,P14812,12S seed storage globulin 2,This is a seed storage protein.,Avena sativa,518,None,MATTRFPSLLFYSYIFLLCNGSMAQLFGQSFTPWQSSRQGGLRGCR...,"[GO:0003674, GO:0045735]",None,"[GO:0003674, GO:0045735]",None,AF-P14812-F1-model_v4.cif,"[IPR022379, IPR006044, IPR014710, IPR006045, I...","{""IPR011051"": [14, 488], ""IPR014710"": [35, 501...",None,NaN
2,A9I8F5,Crossover junction endodeoxyribonuclease RuvC,The RuvA-RuvB-RuvC complex processes Holliday ...,Bordetella petrii (strain ATCC BAA-461 / DSM 1...,182,Cytoplasm,MRVLGIDPGLRRTGFGVIDAEGMRLRYVASGTIVVPPALALPERLK...,"[GO:0008150, GO:0003674, GO:0008152, GO:000998...","[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0005488, GO:0003824, GO:014064...",None,AF-A9I8F5-F1-model_v4.cif,"[IPR036397, IPR012337, IPR002176, IPR020563]","{""IPR012337"": [1, 155], ""IPR036397"": [1, 155],...",94624.Bpet0905,"[murB, fis, dusB, polA, ruvA, ruvA, dapB, purH..."
3,A1TVM0,Uroporphyrinogen decarboxylase,Catalyzes the decarboxylation of four acetate ...,Acidovorax citrulli (strain AAC00-1),376,Cytoplasm,MSFAPLQNDTFLRACRRQATDYTPLWLMRQAGRYLPEYKATRARAG...,"[GO:0008150, GO:0005575, GO:0003674, GO:000815...","[GO:0008150, GO:0008152, GO:0009987, GO:007170...","[GO:0003674, GO:0003824, GO:0016829, GO:001683...","[GO:0005575, GO:0110165, GO:0005737]",AF-A1TVM0-F1-model_v4.cif,"[IPR006361, IPR000257, IPR038071]","{""IPR038071"": [1, 372], ""IPR006361"": [8, 364],...",397945.Aave_4469,"[Aave_0327, Aave_1998, Aave_2103, Aave_4735, h..."
4,Q646B6,Taste receptor type 2 member 13,Receptor that may play a role in the perceptio...,Pan troglodytes,303,Membrane,MESALPSIFTLVIIAEFIIGNLSNGFIVLINCIDWVSKRELSSVDK...,"[GO:0008150, GO:0005575, GO:0003674, GO:000998...","[GO:0008150, GO:0009987, GO:0050896, GO:003250...","[GO:0003674, GO:0060089, GO:0038023, GO:000488...","[GO:0005575, GO:0110165, GO:0016020, GO:0005886]",AF-Q646B6-F1-model_v4.cif,[IPR007960],"{""IPR007960"": [1, 298]}",9598.ENSPTRP00000054728,[GNAT3]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376744,B2IC83,Phosphoribosylformylglycinamidine synthase sub...,Part of the phosphoribosylformylglycinamidine ...,Beijerinckia indica subsp. indica (strain ATCC...,736,Cytoplasm,MSADPPITPELVAAHGLKPDEYQRILTLIGREPSFTELGIFSAMWN...,"[GO:0008150, GO:0005575, GO:0003674, GO:000815...","[GO:0008150, GO:0008152, GO:0009987, GO:007170...","[GO:0003674, GO:0005488, GO:0003824, GO:009715...","[GO:0005575, GO:0110165, GO:0005737]",AF-B2IC83-F1-model_v4.cif,"[IPR010918, IPR016188, IPR010074, IPR041609, I...","{""IPR036921"": [13, 559], ""IPR036676"": [16, 732...",395963.Bind_3119,"[glyA, purC, purN, purM, pheT, guaB, purD, Bin..."
376745,A1CFL8,6-methylcalicylic acide synthase,6-methylsalicylic acid synthase; part of the g...,Aspergillus clavatus (strain ATCC 1007 / CBS 5...,1720,Cytoplasm; Cytosol,MDKQSASGEIPAMRWEPYHRRDPRNAKELSKTTSRGYFLDHLEDFD...,"[GO:0008150, GO:0005575, GO:0003674, GO:000815...","[GO:0008150, GO:0008152, GO:0009987, GO:007170...","[GO:0003674, GO:0005488, GO:0003824, GO:003321...","[GO:0005575, GO:0110165, GO:0005829, GO:0005737]",AF-A1CFL8-F1-model_v4.cif,"[IPR050091, IPR042104, IPR014031, IPR009081, I...","{""IPR016039"": [1, 413], ""IPR001227"": [421, 825...",344612.A1CFL8,"[ACLA_004290, patJ, patL]"
376746,O67289,Ketol-acid reductoisomerase (NADP(+)),Involved in the biosynthesis of branched-chain...,Aquifex aeolicus (strain VF5),333,None,MAKIYYDEDASLDILKDKVIAILGYGSQGHAHALNLRDSGLNVIIG...,"[GO:00

### Testing

In [127]:
import pandas as pd

# Combine all sets into one DataFrame
all_sets = pd.concat([swissprot_train, swissprot_val])
all_sets.reset_index(drop=True, inplace=True)

# Create a set of valid (protein, partner) pairs from all_metadata_df
valid_pairs = set(
    zip(all_metadata_df['swissprot_protein'], all_metadata_df['partner_id'])
)

# Flatten and create (protein, partner) pairs from interaction_partners column
errors = []

for idx, row in all_sets.iterrows():
    protein = row['protein_id']
    partners = row['interaction_partners']

    if not isinstance(partners, list):
        if np.isnan(partners):
            continue
        else:
            errors.append((protein, "interaction_partners not a list"))
            continue

    for partner in partners:
        if (protein, partner) not in valid_pairs:
            errors.append((protein, partner))

# Final test result
if not errors:
    print("✅ All interaction_partners are valid and match all_metadata_df.")
else:
    print(f"❌ Found {len(errors)} mismatches. Examples:")
    print(errors[:10])  # Show first 10 mismatches

✅ All interaction_partners are valid and match all_metadata_df.


In [128]:
critical_cols = ['swissprot_protein', 'partner_id', 'string_protein_id', 'Protein names']
missing = all_metadata_df[critical_cols].isnull().sum()
print("Missing values per column:\n", missing)

Missing values per column:
 swissprot_protein    0
partner_id           0
string_protein_id    0
Protein names        0
dtype: int64


In [131]:
assert all(all_metadata_df['swissprot_protein'].apply(lambda x: isinstance(x, str)))
assert all(all_metadata_df['partner_id'].apply(lambda x: isinstance(x, str)))

In [132]:
dups = all_metadata_df.duplicated(subset=['swissprot_protein', 'partner_id'])
print(f"Number of duplicated interactions: {dups.sum()}")

Number of duplicated interactions: 51824


In [134]:
all_metadata_df = all_metadata_df.drop_duplicates()

In [136]:
all_swissprot_ids = set(pd.concat([swissprot_train, swissprot_val])['protein_id'])
metadata_ids = set(all_metadata_df['swissprot_protein'])

not_in_swissprot = metadata_ids - all_swissprot_ids
print(f"Number of proteins in all_metadata_df not in Swissprot sets: {len(not_in_swissprot)}")

Number of proteins in all_metadata_df not in Swissprot sets: 0


In [137]:
if 'combined_score' in all_metadata_df.columns:
    print(all_metadata_df['combined_score'].describe())
    assert all_metadata_df['combined_score'].between(0, 1000).all()

count    6.569396e+06
mean     8.888915e+02
std      9.620751e+01
min      7.000000e+02
25%      8.050000e+02
50%      9.090000e+02
75%      9.840000e+02
max      9.990000e+02
Name: combined_score, dtype: float64


## Time to Upload

In [138]:
from datasets import Dataset
import json

# Create Hugging Face Datasets
train_hf = Dataset.from_pandas(swissprot_train, preserve_index=False)
val_hf = Dataset.from_pandas(swissprot_val, preserve_index=False)

In [139]:
from datasets import DatasetDict

swissprot_dataset = DatasetDict({
    "train": train_hf,
    "val": val_hf
})

In [140]:
swissprot_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners'],
        num_rows: 376749
    })
    val: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners'],
        num_rows: 19829
    })
})

In [141]:
swissprot_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="swissprot_reasoning",
    commit_message="Edited Swissprot data to reflect PPI metadata"
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/189 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/142M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/189 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/142M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/15.1M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/af5deab369a7627579eda15881949c4c765bda11', commit_message='Edited Swissprot data to reflect PPI metadata', commit_description='', oid='af5deab369a7627579eda15881949c4c765bda11', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [142]:
from datasets import Dataset
import json

# Create Hugging Face Datasets
ppi_metadata = Dataset.from_pandas(all_metadata_df, preserve_index=False)

In [143]:
from datasets import DatasetDict

ppi_dataset = DatasetDict({
    "metadata": ppi_metadata
})

In [144]:
ppi_dataset

DatasetDict({
    metadata: Dataset({
        features: ['swissprot_protein', 'partner_id', 'string_protein_id', 'uniprot_ac', 'uniprot_id', 'identity', 'bit_score', 'coexpression', 'combined_score', 'cooccurence', 'database', 'experimental', 'fusion', 'neighborhood', 'protein2', 'textmining', 'Protein names', 'Function [CC]', 'Subcellular location [CC]', 'Protein names_cleaned', 'length_before', 'length_after'],
        num_rows: 6569396
    })
})

In [145]:
ppi_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="ppi_swissprot_metadata",
    commit_message="Uploading all of the PPI metadata to one central place. Indexed by swissprot_protein and interaction partner"
)

Uploading the dataset shards:   0%|          | 0/6 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1095 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/148M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/1095 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/150M [00:00<?, ?B/s]

...

Uploading...:   0%|          | 0.00/149M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/1095 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/149M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/2ea2c2ad33d24515bd3e2ea42d27bdfc4ad07f75', commit_message='Uploading all of the PPI metadata to one central place. Indexed by swissprot_protein and interaction partner', commit_description='', oid='2ea2c2ad33d24515bd3e2ea42d27bdfc4ad07f75', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Create Efficient Preprocessed PPI info

In [ ]:
#!/usr/bin/env python3
"""
Preprocess CAFA5 experiment_data with PPI information and save as new dataset.

This script loads the experiment_data split, adds efficient PPI preprocessing,
and saves the result as 'experiment_data_with_ppi' configuration.
"""

import os
import sys
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

def create_efficient_ppi_lookup(ppi_metadata: pd.DataFrame, top_n_ppi: int = 10) -> dict:
    """Create ultra-fast PPI lookup with pre-formatted strings."""
    if ppi_metadata.empty:
        return {}
    
    print(f"Creating efficient PPI lookup for {len(ppi_metadata)} PPI entries...")
    
    # Sort entire DataFrame once (much faster than per-group sorting)
    print("Sorting PPI data...")
    sorted_df = ppi_metadata.sort_values(['cafa5_protein', 'combined_score'], ascending=[True, False])
    
    # Group and take top N per protein using vectorized operations
    print("Grouping and selecting top partners...")
    top_partners = sorted_df.groupby('cafa5_protein')['Protein names'].apply(
        lambda x: x.head(top_n_ppi).tolist()
    ).to_dict()
    
    # Pre-format all PPI strings (vectorized string operations)
    print("Pre-formatting PPI strings...")
    ppi_lookup = {}
    for protein_id, partners in tqdm(top_partners.items(), desc="Formatting PPI data"):
        if partners:
            # Format as simple bullet list (no extra processing needed)
            ppi_lookup[protein_id] = '\n'.join(f'- {partner}' for partner in partners)
        else:
            ppi_lookup[protein_id] = ""
    
    print(f"Created PPI lookup for {len(ppi_lookup)} proteins")
    return ppi_lookup


def add_ppi_data_mega_batch(examples_batch: dict, ppi_lookup: dict) -> dict:
    """Add PPI data to a massive batch of examples efficiently."""
    # Get protein IDs for the entire batch
    protein_ids = examples_batch.get("protein_id", [])
    
    print(f"Processing batch of {len(protein_ids)} proteins...")
    
    # Batch lookup all PPI data at once using list comprehension (fastest)
    ppi_formatted = [ppi_lookup.get(pid, "") for pid in protein_ids]
    
    # Add to batch
    examples_batch["ppi_formatted"] = ppi_formatted
    return examples_batch


def main():
    """Main preprocessing function."""
    print("=" * 80)
    print("CAFA5 PPI PREPROCESSING SCRIPT")
    print("=" * 80)
    
    # Configuration
    dataset_name = "wanglab/cafa5"
    cache_dir = "/large_storage/goodarzilab/bioreason/data"
    top_n_ppi = 10
    output_config_name = "experiment_data_with_ppi"
    
    print(f"Using cache directory: {cache_dir}")
    print(f"Processing top {top_n_ppi} PPI partners per protein")
    print(f"Output configuration: {output_config_name}")
    
    # ================================ Load Datasets ================================
    print("\n" + "=" * 80)
    print("LOADING DATASETS")
    print("=" * 80)
    
    # Load experiment data
    print("Loading experiment_data...")
    datasets = load_dataset(dataset_name, name="experiment_data", cache_dir=cache_dir)
    train_dataset = datasets["train"]
    val_dataset = datasets["validation"] if "validation" in datasets else None
    
    print(f"Loaded training dataset: {len(train_dataset)} samples")
    if val_dataset:
        print(f"Loaded validation dataset: {len(val_dataset)} samples")
    
    # Load PPI metadata
    print("Loading PPI metadata...")
    ppi_datasets = load_dataset(dataset_name, name="ppi_metadata", cache_dir=cache_dir)
    ppi_metadata = ppi_datasets["metadata"].to_pandas()
    
    # Validate PPI metadata
    required_columns = ["cafa5_protein", "Protein names", "combined_score"]
    missing_columns = [col for col in required_columns if col not in ppi_metadata.columns]
    if missing_columns:
        raise ValueError(f"PPI metadata is missing required columns: {missing_columns}")
    
    print(f"Loaded PPI metadata: {len(ppi_metadata)} entries")
    
    # ================================ Create PPI Lookup ================================
    print("\n" + "=" * 80)
    print("CREATING PPI LOOKUP")
    print("=" * 80)
    
    ppi_lookup = create_efficient_ppi_lookup(ppi_metadata, top_n_ppi)
    
    # ================================ Process Training Data ================================
    print("\n" + "=" * 80)
    print("PROCESSING TRAINING DATA")
    print("=" * 80)
    
    print("Adding PPI data to training dataset...")
    train_dataset_with_ppi = train_dataset.map(
        add_ppi_data_mega_batch,
        batched=True,
        batch_size=1_000_000,  # Process entire dataset in one go if possible
        desc="Adding PPI data to training set",
        fn_kwargs={"ppi_lookup": ppi_lookup},
    )
    
    # ================================ Process Validation Data ================================
    val_dataset_with_ppi = None
    if val_dataset:
        print("\n" + "=" * 80)
        print("PROCESSING VALIDATION DATA")
        print("=" * 80)
        
        print("Adding PPI data to validation dataset...")
        val_dataset_with_ppi = val_dataset.map(
            add_ppi_data_mega_batch,
            batched=True,
            batch_size=1_000_000,  # Process entire dataset in one go if possible
            desc="Adding PPI data to validation set",
            fn_kwargs={"ppi_lookup": ppi_lookup},
        )
    
    # ================================ Verify Results ================================
    print("\n" + "=" * 80)
    print("VERIFYING RESULTS")
    print("=" * 80)
    
    # Check some samples to ensure PPI data was added correctly
    sample_with_ppi = 0
    sample_without_ppi = 0
    
    for i, sample in enumerate(train_dataset_with_ppi.select(range(min(100, len(train_dataset_with_ppi))))):
        if 'ppi_formatted' in sample and sample['ppi_formatted'].strip():
            sample_with_ppi += 1
        else:
            sample_without_ppi += 1
    
    print(f"Verification (first 100 samples):")
    print(f"  - Samples with PPI data: {sample_with_ppi}")
    print(f"  - Samples without PPI data: {sample_without_ppi}")
    
    # Show example
    for sample in train_dataset_with_ppi.select(range(10)):
        if 'ppi_formatted' in sample and sample['ppi_formatted'].strip():
            print(f"\n📋 Example PPI data for protein {sample.get('protein_id', 'unknown')}:")
            ppi_preview = sample['ppi_formatted'][:150] + "..." if len(sample['ppi_formatted']) > 150 else sample['ppi_formatted']
            print(ppi_preview)
            break
    
    # ================================ Save to Hugging Face ================================
    print("\n" + "=" * 80)
    print("SAVING TO HUGGING FACE")
    print("=" * 80)
    
    try:
        # Push training data
        print(f"Pushing training dataset to hub as '{output_config_name}'...")
        train_dataset_with_ppi.push_to_hub(
            dataset_name, 
            config_name=output_config_name, 
            split="train"
        )
        print("✓ Training data pushed successfully!")
        
        # Push validation data if it exists
        if val_dataset_with_ppi:
            print(f"Pushing validation dataset to hub as '{output_config_name}'...")
            val_dataset_with_ppi.push_to_hub(
                dataset_name, 
                config_name=output_config_name, 
                split="validation"
            )
            print("✓ Validation data pushed successfully!")
        
        print(f"\n🎉 SUCCESS! Dataset saved as '{dataset_name}' with config_name='{output_config_name}'")
        print(f"Usage example:")
        print(f"  from datasets import load_dataset")
        print(f"  ds = load_dataset('{dataset_name}', name='{output_config_name}')")
        print(f"  train_data = ds['train']")
        
    except Exception as e:
        print(f"❌ Error pushing to hub: {e}")
        print("You may need to login with `huggingface-cli login` first")
        return False
    
    # ================================ Summary Statistics ================================
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    
    # Get all unique proteins from both train and validation sets
    all_dataset_proteins = set(train_dataset_with_ppi['protein_id'])
    if val_dataset_with_ppi:
        all_dataset_proteins.update(val_dataset_with_ppi['protein_id'])
    
    # Count proteins in the dataset that actually have PPI data
    proteins_with_ppi = [p for p in all_dataset_proteins if p in ppi_lookup and ppi_lookup[p]]
    
    total_proteins = len(all_dataset_proteins)
    total_ppi_proteins = len(proteins_with_ppi)
    
    print(f"Dataset statistics:")
    print(f"  - Total training samples: {len(train_dataset_with_ppi)}")
    if val_dataset_with_ppi:
        print(f"  - Total validation samples: {len(val_dataset_with_ppi)}")
    print(f"  - Unique proteins (train+val): {total_proteins}")
    print(f"  - Proteins with PPI data: {total_ppi_proteins}")
    print(f"  - PPI coverage: {total_ppi_proteins/total_proteins*100:.1f}%")
    print(f"  - Top N PPI partners: {top_n_ppi}")
    
    print("\n" + "=" * 80)
    print("PREPROCESSING COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    
    return True


if __name__ == "__main__":
    success = main()
    sys.exit(0 if success else 1)